# Detección de anomalías en paneles solares con YOLOv5

**Actividad 3 - Aprendizaje Profundo para Percepción y Control**

Este notebook documenta el fine-tuning de YOLOv5 para detectar cuatro estados visibles en paneles solares: panel cubierto (`cover`), grieta o rotura (`crack`), suciedad o polvo (`dust`) y panel normal (`normal`). Está organizado según los apartados de la rúbrica.


## 1. Definición del problema

La inspección manual de instalaciones fotovoltaicas consume tiempo y puede retrasar la identificación de anomalías que reducen la producción o requieren mantenimiento. Se propone entrenar un detector de objetos capaz de localizar y clasificar visualmente cuatro condiciones de paneles solares.

El modelo puede servir como apoyo preliminar a inspecciones realizadas con cámaras terrestres o drones. No sustituye una inspección técnica, porque algunas fallas eléctricas o térmicas no son visibles en imágenes RGB.


## 2. Creación y preparación del dataset

El dataset procede de Roboflow Universe y se distribuye con licencia CC BY 4.0. Las anotaciones originales están en formato YOLO, compatible con YOLOv5.

Para evitar fuga de información, las variantes aumentadas derivadas de una misma imagen fuente se mantuvieron juntas en una única partición. Se aplicó una división reproducible aproximada de 70 % entrenamiento, 20 % validación y 10 % prueba usando la semilla 42.

Fuente: https://universe.roboflow.com/workshop-pydqh/solar-panel-0swal-vrqxd-seo7f/dataset/1


In [ ]:
# CONFIGURACIÓN LOCAL
# Abre el notebook mediante iniciar_notebook_local.bat para que estas rutas
# se resuelvan automáticamente desde la carpeta ProyectoYOLOv5.
from pathlib import Path
import os
import subprocess
import sys

PROJECT_DIR = Path.cwd().resolve()
YOLOV5_DIR = PROJECT_DIR / "vendor" / "yolov5"
PYTHON_EXECUTABLE = Path(sys.executable)
DATA_DIR = PROJECT_DIR / "data" / "solar_panels"

print("Python:", PYTHON_EXECUTABLE)
print("Proyecto:", PROJECT_DIR)
print("YOLOv5:", YOLOV5_DIR)
print("Dataset:", DATA_DIR)


### 2.1 Configuración local

El proyecto está preparado para Windows. Inicia Jupyter con `iniciar_notebook_local.bat` desde la carpeta raíz. De esta forma, el notebook utiliza automáticamente:

- El entorno virtual local `.venv`.
- El repositorio incluido en `vendor/yolov5`.
- El dataset situado en `data/solar_panels`.

No es necesario editar rutas para la ejecución local.


In [ ]:
# Validación del entorno y generación del YAML con rutas locales.
assert PROJECT_DIR.name == "ProyectoYOLOv5", (
    "Abre Jupyter con iniciar_notebook_local.bat desde la raíz del proyecto."
)
assert DATA_DIR.is_dir(), f"No se encuentra el dataset: {DATA_DIR}"
assert (YOLOV5_DIR / "train.py").is_file(), f"No se encuentra YOLOv5: {YOLOV5_DIR}"

RUNTIME_YAML = PROJECT_DIR / "solar_panels_runtime.yaml"
RUNTIME_YAML.write_text(
    f"""path: {DATA_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 4
names: [cover, crack, dust, normal]
""",
    encoding="utf-8",
)
print(RUNTIME_YAML.read_text(encoding="utf-8"))


### 2.2 Verificación del dataset


In [ ]:
import json
import subprocess
import sys

report_path = PROJECT_DIR / "reports" / "dataset_report.json"
subprocess.run(
    [sys.executable, str(PROJECT_DIR / "scripts" / "validate_dataset.py"),
     "--dataset", str(DATA_DIR), "--output", str(report_path)],
    check=True,
)
report = json.loads(report_path.read_text(encoding="utf-8"))
report


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random

class_names = ["cover", "crack", "dust", "normal"]
colors = ["#ff9800", "#f44336", "#795548", "#4caf50"]

def show_annotated(image_path):
    label_path = Path(str(image_path).replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)).with_suffix(".txt")
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for line in label_path.read_text().splitlines():
        cls, xc, yc, bw, bh = map(float, line.split())
        cls = int(cls)
        x1, y1 = (xc - bw / 2) * width, (yc - bh / 2) * height
        x2, y2 = (xc + bw / 2) * width, (yc + bh / 2) * height
        draw.rectangle((x1, y1, x2, y2), outline=colors[cls], width=3)
        draw.text((x1 + 3, y1 + 3), class_names[cls], fill=colors[cls])
    return image

samples = random.Random(42).sample(list((DATA_DIR / "images" / "train").glob("*")), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, path in zip(axes.flat, samples):
    ax.imshow(show_annotated(path))
    ax.set_title(path.name[:35])
    ax.axis("off")
plt.tight_layout()


## 3. Proceso de entrenamiento del modelo

Se utiliza YOLOv5s con pesos preentrenados en COCO. Esta variante ofrece un equilibrio adecuado entre velocidad, consumo de memoria y precisión para una práctica académica. El entrenamiento se realiza durante 50 épocas, con imágenes de 640 píxeles y `batch_size=16`. Si la GPU no dispone de memoria suficiente, se puede reducir el lote a 8.


In [ ]:
# Comprobación de las dependencias instaladas en el entorno local.
required_modules = [
    "torch", "torchvision", "cv2", "numpy", "pandas",
    "matplotlib", "yaml", "PIL",
]
missing_modules = []
for module_name in required_modules:
    try:
        __import__(module_name)
    except ImportError:
        missing_modules.append(module_name)

if missing_modules:
    raise RuntimeError(
        "Faltan dependencias: "
        + ", ".join(missing_modules)
        + ". Ejecuta instalar_entorno_local.bat."
    )
print("Dependencias locales verificadas correctamente.")


In [ ]:
import torch

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo seleccionado:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Este equipo entrenará en CPU. El proceso completo será lento.")


In [ ]:
# Parámetros editables del entrenamiento completo.
IMG_SIZE = 640
BATCH_SIZE = 16       # En CPU puedes reducirlo si el equipo se queda sin memoria.
EPOCHS = 50
RUN_NAME = "solar_panels_yolov5s"
BASE_WEIGHTS = PROJECT_DIR / "yolov5s.pt"

assert BASE_WEIGHTS.exists(), (
    f"No se encuentran los pesos base: {BASE_WEIGHTS}. "
    "Ejecuta probar_proyecto_local.bat para descargarlos y verificar el entorno."
)

train_command = [
    str(PYTHON_EXECUTABLE), str(YOLOV5_DIR / "train.py"),
    "--img", str(IMG_SIZE),
    "--batch", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--data", str(RUNTIME_YAML),
    "--weights", str(BASE_WEIGHTS),
    "--project", str(PROJECT_DIR / "runs" / "train"),
    "--name", RUN_NAME,
    "--device", DEVICE,
    "--cache",
]
print("Ejecutando:", subprocess.list2cmdline(train_command))
subprocess.run(train_command, cwd=YOLOV5_DIR, check=True)


## 4. Informe de resultados del modelo

Esta sección debe ejecutarse después del entrenamiento. Primero se presentan los resultados objetivos y después se realiza una interpretación crítica, tal como exige la rúbrica.


In [ ]:
from IPython.display import display
import pandas as pd

RUN_DIR = PROJECT_DIR / "runs" / "train" / "solar_panels_yolov5s"
results = pd.read_csv(RUN_DIR / "results.csv")
results.columns = results.columns.str.strip()
display(results.tail())

metric_columns = [c for c in results.columns if any(k in c for k in ("precision", "recall", "mAP_0.5", "mAP_0.5:0.95"))]
display(results[metric_columns].tail(1))


In [ ]:
from IPython.display import Image as DisplayImage, display

for filename in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png"]:
    path = RUN_DIR / filename
    if path.exists():
        print(filename)
        display(DisplayImage(filename=str(path)))


In [ ]:
# Evaluación independiente sobre el conjunto de prueba.
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
assert BEST_WEIGHTS.exists(), f"No se encontraron los pesos: {BEST_WEIGHTS}"

validation_command = [
    str(PYTHON_EXECUTABLE), str(YOLOV5_DIR / "val.py"),
    "--weights", str(BEST_WEIGHTS),
    "--data", str(RUNTIME_YAML),
    "--img", str(IMG_SIZE),
    "--task", "test",
    "--project", str(PROJECT_DIR / "runs" / "val"),
    "--name", "test_evaluation",
    "--device", DEVICE,
    "--save-txt",
    "--save-conf",
]
print("Ejecutando:", subprocess.list2cmdline(validation_command))
subprocess.run(validation_command, cwd=YOLOV5_DIR, check=True)


### 4.1 Interpretación crítica

Completar tras ejecutar el entrenamiento:

- **Precisión:** indicar el valor obtenido y explicar qué proporción de las detecciones fue correcta.
- **Recall:** indicar el valor y analizar cuántas anomalías reales quedaron sin detectar.
- **mAP@0.5 y mAP@0.5:0.95:** comparar ambos valores y explicar el efecto de exigir una localización más precisa.
- **Resultados por clase:** identificar las clases más y menos fiables mediante la matriz de confusión y la curva PR.
- **Idoneidad:** valorar si el desempeño es suficiente como sistema de apoyo y explicar sus limitaciones.

Un resultado global alto no basta: es especialmente importante comprobar que `crack` y `dust` mantienen un recall adecuado, porque los falsos negativos en anomalías pueden retrasar intervenciones de mantenimiento.


## 5. Inferencia del modelo


In [ ]:
# Inferencia sobre imágenes reservadas del conjunto de prueba.
TEST_IMAGES = DATA_DIR / "images" / "test"
DETECTION_RUN = PROJECT_DIR / "runs" / "detect" / "test_predictions"

detection_command = [
    str(PYTHON_EXECUTABLE), str(YOLOV5_DIR / "detect.py"),
    "--weights", str(BEST_WEIGHTS),
    "--img", str(IMG_SIZE),
    "--conf", "0.25",
    "--source", str(TEST_IMAGES),
    "--project", str(PROJECT_DIR / "runs" / "detect"),
    "--name", "test_predictions",
    "--device", DEVICE,
    "--save-txt",
    "--save-conf",
]
print("Ejecutando:", subprocess.list2cmdline(detection_command))
subprocess.run(detection_command, cwd=YOLOV5_DIR, check=True)


In [ ]:
from IPython.display import Image as DisplayImage, display

prediction_files = [
    path
    for path in DETECTION_RUN.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

for path in sorted(prediction_files)[:12]:
    display(DisplayImage(filename=str(path), width=700))


### 5.1 Comentario de las inferencias

Describir con ejemplos concretos:

- Detecciones correctas y nivel de confianza.
- Falsos positivos o clases confundidas.
- Objetos omitidos por el modelo.
- Influencia de iluminación, perspectiva, distancia y tamaño de la anomalía.
- Posibles mejoras: más imágenes originales, revisión de etiquetas, balance de clases y captura de escenarios reales adicionales.


## 6. Conclusiones

Completar con los valores finales. La conclusión debe responder si el detector resuelve razonablemente el problema planteado, cuáles son sus clases más fiables, qué errores siguen siendo relevantes y qué acciones permitirían llevarlo a un entorno real.

Debe aclararse que el dataset incluye aumentos generados previamente y que la partición utilizada en este notebook agrupa todas las variantes de una misma fuente para reducir una evaluación excesivamente optimista.


## 7. Referencias

- Ultralytics. Repositorio oficial de YOLOv5: https://github.com/ultralytics/yolov5
- Dataset Solar Panel, Roboflow Universe, licencia CC BY 4.0: https://universe.roboflow.com/workshop-pydqh/solar-panel-0swal-vrqxd-seo7f/dataset/1
- Jocher, G. et al. YOLOv5 documentation and source code.
